## Setup and start

In [27]:
# Set up the OpenAI client:

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [28]:
# Load the data and build the search index - using python files, ??rewriting database??:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [29]:
# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [30]:
# Let's try a question:

assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and start a model locally:\n```bash\nollama run llama3\n```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test the local server:\n```bash\ncurl http://localhost:11434\n```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```'

In [31]:
# Now try something slightly different:

assistant.rag("How do I run Olama locally?")

# Notice we did a typo here - the word "Olama" doesn't match "Ollama" in our index. 
# We use lexical search, so it looks for the exact word and finds nothing. 
# The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

# This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. 
# The pipeline doesn't know the search failed, so it can't try again with a corrected query.

# LLM ANSWER << 'I don’t see any FAQ entry in the context about running Olama locally.
# \n\nIf you meant a different tool from the course context, I can help with that.' >>


'I don’t see any FAQ entry about **Olama/Ollama** specifically.\n\nThe closest related FAQ says you **can run the course locally instead of Codespaces** if you’re comfortable setting up the needed tools such as **Python, `uv`, Jupyter, Docker, and any other module-specific tools**.\n\nIf you meant running the course environment locally, the FAQ says to:\n- set up the required tools locally,\n- **document your setup**, and\n- keep the environment **reproducible**.\n\nIf you meant something else by “Olama,” send the exact name and I’ll check again.'

## Function calling

In [32]:
# Why our RAG system from above cannot manage typos?
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

#  =========== The agent alternative ============
# An agent puts the LLM in charge.

# Instead of running search ourselves, we give the LLM a search tool. It decides when to call it and what to search for.
# The LLM searched, saw the results were bad, and decided to try again with a different query. It made that decision on its own. 
# We didn't write any code to handle typos.

# The difference is about who makes the decisions:

# With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
# With an agent, the LLM decides. It chooses which actions to take and when to stop.
# The mechanism that makes this possible is function calling, and that's what the rest of this part is about.

# Defining the tool
# First we define a top-level search function that queries the index directly. The model will reference it by this name. 
# We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [33]:
# Asking without tools
# Let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

# The model answers from its general knowledge, something like "it depends on the course" or "check the course website". 
# It doesn't know about our FAQ, so the answer is vague and not helpful. 
# This is exactly why we need RAG, and why we want to hand the model a tool.
    

'Absolutely — in many cases you can still join, but it depends on the course’s enrollment rules and whether registration is still open.\n\nTo check quickly:\n1. **Look at the course page** for the enrollment deadline or “open enrollment” status.\n2. **Contact the instructor or course admin** and ask if late enrollment is allowed.\n3. If it’s a **live cohort**, ask whether there’s a waitlist or next start date.\n4. If it’s **self-paced**, you can often join anytime.\n\nIf you want, I can help you draft a short message to ask about joining.'

In [34]:
# Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the 
# function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we 
# describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# The description is the most important field, because the model reads it to decide when to call the function. 
# parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.


In [35]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [36]:
# Sending the question with the tool
# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

# Look at the output. Instead of a message with the answer, the response contains a function_call entry.
# The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first.

# Look at the arguments too. The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with. 
# So it rewrote our enrollment question into search keywords like "enroll late join course".



[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after course started FAQ"}', call_id='call_sLj8c087xc39M2USk0fbf5Wb', name='search', type='function_call', id='fc_058d16c9c4b6f566006a35af2d5ce881a29f89415d49a07417', namespace=None, status='completed')]

In [37]:
messages
# original question - 'I just discovered the course. Can I join it?'
# LLM searched it as - "join course discovered course can I join enrollment late registration eligibility"

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]

In [38]:
# Executing the function and sending the result back
# The function call contains JSON arguments. We parse them, call our search function, and serialize the result.

import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [39]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I sta

In [40]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "answer": "No, you can only get a certificate if you finish the course with a \\"live\\" cohort.\\n\\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\\n\\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n   

In [41]:
# Now we send this result back to the model. First, we add the model's output to the conversation 
# history - the model needs to see its own function call. Then we add the tool result.

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join after course started FAQ"}', call_id='call_sLj8c087xc39M2USk0fbf5Wb', name='search', type='function_call', id='fc_058d16c9c4b6f566006a35af2d5ce881a29f89415d49a07417', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_sLj8c087xc39M2USk0fbf5Wb',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get

In [42]:
# The call_id links the tool result to the specific function call the model requested. If the model makes multiple function 
# calls in one turn, each one gets its own call_id.

# Asking the model again
# We call the API a second time with the expanded history:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text


# This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper 
# course-specific answer.

# We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. 
# If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. 
# That means the question, the decision to call search, and the result we got back.

    

'Yes — you can still join and start learning now.\n\nIf you want a certificate, though, you’ll need to finish while the course is still “live” and submit your project before submissions close. If you’re just learning for yourself, you can follow the materials anytime.'

In [43]:
# That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. 
# Turning RAG agentic means more round-trips.

# People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. 
# The LLM decides which tools to call.

# Token usage and cost
# We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

# The response has a usage field with the token counts:

usage = response.usage
usage.input_tokens, usage.output_tokens



(773, 59)

In [44]:
# For each model the provider publishes a price per million input tokens and per million output tokens. 
# Plug those numbers in to convert tokens to dollars.

def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

# This usage is only for the second API call. The first call also has its own usage and its own cost. 
# That was the call where the model decided to invoke search. Two calls means we pay twice. 
# We pay even more on the second call, because we resend the full history as input.



Total cost: $ 0.0001176


## The agentic loop

In [45]:

# Anatomy of an agent
# With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

# An agent has three parts:

# -- Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
# -- Tools, the functions the agent can call to carry out the task. For us that's only search.
# -- Memory, the message history. We append every prompt, every model output, and every tool result. 
# The agent reads this to know what it has already tried.
# Everything below is the code that wires these three together inside a loop.

# A developer prompt
# So far we've relied on the model to figure out when to search. We make that more reliable with a developer 
# message that spells out how to behave. This is where we give the agent its role. The same message also pushes 
# it toward multiple searches, so we get to watch the loop run more than once.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [46]:
# A function-call helper
# We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper.
# It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. 
# We only have one tool for now, so we dispatch on the function name directly.

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }
    
# The helper returns the exact structure the Responses API expects. 
# When we add more tools later, we'll extend this with more if branches (or switch to a registry).


In [47]:

# Processing one response
# Let's process a single model response. We append each output entry to the conversation, 
# print any messages, and run any function calls. Function-call results get appended too.

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)
        
# The has_function_calls flag tells us whether the model needs another API call. 
# If the response contains a function call, the updated messages has tool output the model hasn't seen yet. 
# We'll need to send it back.


function_call: search {"query":"join course discovered course can I join enrollment registration FAQ"}
function_call: search {"query":"late join course signup enroll after course start FAQ"}


In [48]:
# The full agent loop
# We wrap this in a while loop. The loop keeps calling the model until it returns a response without any function calls. 
# We also keep an iteration counter so we can see how many round-trips happened.

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

        
# This is the core agent loop. The model reasons about the next action. Your code performs it, 
# and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

# We don't decide how many times the model searches. The model does, and we keep looping until it stops asking for tools.

# The exit condition is the simplest one possible. No function calls this turn means we're done. 
# Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. 
# You might cap it at five iterations and force an answer on the last one. The core is still this one flag.


iteration #1...
ASSISTANT:
Yes — you can still join. You can start learning and working through the materials even if you just discovered the course.

One caveat: if you want a certificate, you need to submit your project while submissions are still open, and certificates are only available for the live cohort, not self-paced mode.

If you want, I can also help you figure out the best way to get started with the course. Are there other areas you want to explore?


In [49]:
# Wrapping it in a function
# Let's wrap the loop in a function so we can reuse it. The function takes the instructions 
# and the question as parameters, and returns the final answer.


def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

    

In [50]:
# Try it with a question that has a typo:

agent_loop(instructions, "How do I run Olama locally?")
    
# Watch what happens. The agent searches for "Olama" and gets poor results. It then searches again with "Ollama" and finds the answer. 
# The loop lets the model recover from a bad search on its own. That's the whole point of going agentic.


iteration #1...
function_call: search {"query":"Ollama local run install start local model FAQ"}
function_call: search {"query":"run Ollama locally model pull serve FAQ"}
function_call: search {"query":"Ollama localhost command line FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model, start it locally, and open a chat prompt.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model, start it locally, and open a chat prompt.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, you may need to restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n

In [51]:
# Also try the course enrollment question:

agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered the course can I still join"}
function_call: search {"query":"late registration enrollment course join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you discovered it late.

If your goal is a certificate, the key thing is to submit your project while submissions are still open. You can also start learning from the videos, notebooks, and GitHub materials at any time.

If you want, I can also help with:
- how to start from week 1 or catch up quickly
- whether you can still get a certificate
- how homework and deadlines work


'Yes — you can still join the course even if you discovered it late.\n\nIf your goal is a certificate, the key thing is to submit your project while submissions are still open. You can also start learning from the videos, notebooks, and GitHub materials at any time.\n\nIf you want, I can also help with:\n- how to start from week 1 or catch up quickly\n- whether you can still get a certificate\n- how homework and deadlines work'

In [52]:
# Encouraging multiple searches
# There's a subtle issue here. The model often answers after the first search, 
# even when more searches would help. It reasons that it already knows enough, so why bother. 
# We push it to explore more by rewriting the instructions.

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

# Now the agent makes multiple searches per question and doesn't stop after the first round of results. 
# The instructions are how we steer the agent. It can still decide to skip ahead sometimes, 
# so don't expect it to follow them every single run.



iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

If you want a certificate, the key thing is to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced study.

If you’d like, I can also help you figure out how to start the course or whether you can still get a certificate this round.


'Yes — you can still join the course even if you just discovered it.\n\nIf you want a certificate, the key thing is to submit your project while submissions are still being accepted. Also, certificates are only available for the live cohort, not self-paced study.\n\nIf you’d like, I can also help you figure out how to start the course or whether you can still get a certificate this round.'